In [4]:
import numpy as np
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

# 1. Загрузка исходной структуры из вашего CIF-файла
raw_structure = Structure.from_file("Vampire_data/Ti2O3.cif")

# Находим стандартную условную ячейку (Conventional Standard)
analyzer = SpacegroupAnalyzer(raw_structure)
structure = analyzer.get_conventional_standard_structure()

# 2. Переводим решетку в строгую декартову (ортогональную) систему для VAMPIRE
# Берём реальные размеры ячейки (a, b, c) из CIF
abc = structure.lattice.abc

# Принудительно создаем идеальную единичную матрицу векторов
lattice_matrix = np.array(
    [[abc[0], 0.0, 0.0], [0.0, abc[1], 0.0], [0.0, 0.0, abc[2]]]
)

# 3. Обработка атомов и их подрешеток
ucf_atoms = []
ti_count = 0

for i, site in enumerate(structure):
    element = site.specie.symbol

    # Извлекаем АБСОЛЮТНЫЕ декартовы координаты атома в Ангстремах (site.coords)
    cart_coords = site.coords

    # Проецируем их на нашу новую строго положительную ортогональную ячейку
    new_frac_coords = cart_coords / np.array(abc)

    # Приводим строго к диапазону [0.0, 1.0) по периодическим граничным условиям
    new_frac_coords = new_frac_coords % 1.0

    # Разделение по магнитным материалам
    if element == "Ti":
        if ti_count % 2 == 0:
            mat_category = 1  # Ti_A (Спин вверх)
        else:
            mat_category = 2  # Ti_B (Спин вниз)
        ti_count += 1
    elif element == "O":
        mat_category = 3  # Кислород (Немагнитный)
    else:
        mat_category = 4

    ucf_atoms.append(
        (i, new_frac_coords, new_frac_coords, new_frac_coords, mat_category)
    )

num_atoms = len(ucf_atoms)
num_interactions = 0

# 4. Запись абсолютно корректного файла Ti2O3.ucf
with open("Vampire_data/Ti2O3.ucf", "w") as f:
    f.write("# Unit cell file for Ti2O3 (Orthogonalized for VAMPIRE)\n")

    # Строка 1: Размеры элементарной ячейки
    f.write(f"{abc[0]:.6f} {abc[1]:.6f} {abc[2]:.6f}\n")

    # Строки 2-4: Идеальные базисные векторы (никаких минусов или синусов)
    f.write("1.000000 0.000000 0.000000\n")
    f.write("0.000000 1.000000 0.000000\n")
    f.write("0.000000 0.000000 1.000000\n")

    # Строка 5: Кол-во атомов и взаимодействий
    f.write(f"{num_atoms} {num_interactions}\n")

    # Строки с 6: Координаты атомов
    f.write("# ID  x_frac  y_frac  z_frac  mat_id\n")
    for atom in ucf_atoms:
        f.write(
            f"{atom} {atom:.6f} {atom:.6f} {atom:.6f} {atom}\n"
        )

print("Файл Ti2O3.ucf успешно сгенерирован!")


TypeError: unsupported format string passed to tuple.__format__

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Загрузка данных из файла output
# VAMPIRE использует пробелы/табуляцию в качестве разделителей.
# Скрипт автоматически найдет строку с заголовками, если она начинается с '#'
file_path = "Vampire_data/output"

try:
    # Считываем данные, определяя разделители как любое количество пробелов
    data = pd.read_csv(file_path, sep=r"\s+", comment="#", header=None)

    # Присваиваем имена колонкам в соответствии со структурой VAMPIRE
    # Измените порядок или названия, если ваш файл содержит другое число материалов
    columns = [
        "Temperature",
        "Total_Mag",
        "Ti_A_Mag",
        "Ti_B_Mag",
        "O_Mag",
        "Susceptibility",
    ]
    data.columns = columns[: len(data.columns)]

except Exception as e:
    print(f"Ошибка при чтении файла: {e}")
    print("Убедитесь, что файл 'output' находится в одной папке со скриптом.")
    exit()

# 2. Создание графика
plt.figure(figsize=(9, 6), dpi=100)

# Строим графики намагниченности для подрешетки А и Б
plt.plot(
    data["Temperature"],
    data["Ti_A_Mag"],
    label="Подрешетка Ti_A (Spin Up)",
    color="#e41a1c",
    linewidth=2,
    marker="o",
    markersize=4,
)
plt.plot(
    data["Temperature"],
    data["Ti_B_Mag"],
    label="Подрешетка Ti_B (Spin Down)",
    color="#377eb8",
    linewidth=2,
    marker="s",
    markersize=4,
)

# Опционально: можно вывести общую намагниченность (для АФМ она должна быть близка к 0)
# plt.plot(data['Temperature'], data['Total_Mag'], label='Полная намагниченность', color='black', linestyle='--')

# 3. Оформление графика
plt.title(
    "Температурная зависимость намагниченности подрешеток $Ti_2O_3$",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Температура $T$ (K)", fontsize=12)
plt.ylabel("Нормированная намагниченность $M / M_0$", fontsize=12)

# Сетка и легенда
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11, loc="upper right")

# Ограничения осей для красивого отображения
plt.xlim(0, data["Temperature"].max() * 1.05)
plt.ylim(-0.05, 1.05)

# Инструкция для исследователя на графике
plt.text(
    data["Temperature"].max() * 0.1,
    0.1,
    "Точка, где намагниченность падаёт до 0\nявляется температурой Нееля ($T_N$)",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="gray"),
    fontsize=10,
)

# Tight layout предотвращает обрезание подписей
plt.tight_layout()

# Сохранение и показ графика
plt.savefig("Ti2O3_magnetisation_vs_temp.png", dpi=300)
plt.show()

print(
    "График успешно построен и сохранен как 'Ti2O3_magnetisation_vs_temp.png'!"
)
